[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C29_Frontier_Interp_Course/01_sae/01_sae.ipynb)

# 01 · 稀疏自编码器 SAE（从零训练，对拍真特征）

目标：在**合成叠加数据**（真特征已知）上，用 numpy 从零实现并训练 **L1 / TopK** SAE（含 Adam 优化器），并对拍它们**恢复了多少真特征**。

路线：合成数据 → SAE 前向 → L1 SAE 从零训(Adam) + 恢复率 → TopK SAE → 重建-稀疏帕累托前沿 → dead features 复活 → ✏️ 练习 ×4 → 📖 答案 → 🧪 真实 SAE 配置胶囊。

> 心智模型：**SAE = 特征解压器**。编码器求稀疏系数，解码器各列 = 字典原子(特征方向, 单位范数)。损失 = 重建 + 稀疏。

## 1 · 合成叠加数据 + SAE 前向

先造数据：`d` 维空间塞 `m_true > d` 个**真特征**，每输入稀疏激活少数个、线性叠加、加噪。我们知道真特征方向 `F_true` 与真激活码 `S`（ground truth）。

再定义 SAE 前向：编码 `f = ReLU(W_e (x - b_d) + b_e)`，解码 `x̂ = W_d f + b_d`。解码器列单位化 = 特征方向。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def make_data(n, d=20, m_true=40, p_active=0.04, amp=(0.6, 1.4), noise=0.01, seed=0):
    '''返回 X[n,d] 激活, S[n,m_true] 真激活码, F[m_true,d] 真特征方向(单位行)。
       m_true>d 故必然叠加；每输入平均激活 ~ p_active*m_true 个特征。'''
    r = np.random.default_rng(seed)
    F = r.standard_normal((m_true, d)); F /= np.linalg.norm(F, axis=1, keepdims=True)
    S = (r.random((n, m_true)) < p_active).astype(float)
    S *= r.uniform(*amp, size=(n, m_true))
    X = S @ F + noise * r.standard_normal((n, d))
    return X, S, F

X, S_true, F_true = make_data(8000, d=20, m_true=40)
L0_true = S_true.astype(bool).sum(1).mean()
print(f'X{X.shape}  真特征 m_true={F_true.shape[0]} > d={X.shape[1]} (叠加)  真L0≈{L0_true:.2f}')
assert F_true.shape[0] > X.shape[1]
assert abs(np.linalg.norm(F_true[0]) - 1) < 1e-9
print('✅ 合成叠加数据就绪（真相已知）')

In [ ]:
def sae_forward(x, We, be, Wd, bd, activation='relu', K=None, theta=None):
    '''SAE 前向。x:[n,d]; We:[m,d]; Wd:[d,m]. 返回 (f[n,m], xhat[n,d]).'''
    pre = (x - bd) @ We.T + be              # [n,m] 编码前值
    if activation == 'relu':
        f = np.maximum(pre, 0.0)
    elif activation == 'topk':
        f = np.maximum(pre, 0.0)
        if K is not None and K < f.shape[1]:
            kth = np.partition(f, -K, axis=1)[:, -K][:, None]   # 每行第K大
            f = np.where(f >= kth, f, 0.0)                      # 只留 >= 第K大
    elif activation == 'jumprelu':
        f = np.where(pre > theta, pre, 0.0)                     # 阈值跳变
    else:
        raise ValueError(activation)
    xhat = f @ Wd.T + bd                    # [n,d] 重建
    return f, xhat

# 随机初始化一个 SAE，验证维度与重建误差是个有限正数
d, m = 20, 80                               # 过完备 4x
We0 = rng.standard_normal((m, d)) * 0.1
Wd0 = rng.standard_normal((d, m)); Wd0 /= np.linalg.norm(Wd0, axis=0, keepdims=True)
be0 = np.zeros(m); bd0 = X.mean(0)
f0, xh0 = sae_forward(X[:64], We0, be0, Wd0, bd0)
print('f', f0.shape, '| xhat', xh0.shape, '| 初始重建MSE', np.mean((X[:64]-xh0)**2).round(4))
assert f0.shape == (64, m) and xh0.shape == (64, 20)
assert (f0 >= 0).all(), 'ReLU 特征应非负'
print('✅ SAE 前向维度正确，特征非负')

## 2 · 从零训 L1 SAE（Adam），并对拍恢复了多少真特征

用全 numpy 手写梯度 + **Adam** 训 L1 SAE。损失 `L = ‖x-x̂‖² + λ‖f‖₁`（解码器列保持单位范数，故 L1 直接作用于 f）。

> **为什么 Adam 而非 SGD？** SAE 的损失面病态，朴素 SGD 收敛极慢、恢复不出干净特征。Adam（逐参数自适应步长）让玩具也能恢复出真方向。这本身就是「怎么训 SAE」的一条实操教训。

训完做**特征恢复**对拍：把学到的字典原子(解码器列)与真特征 `F_true` 做二部匹配，看多少真特征被高余弦恢复——真模型上做不到、合成数据独有的金标准。

In [ ]:
def train_sae(X, m, lr=0.002, lam=0.3, steps=4000, activation='relu', K=None,
              batch=512, seed=1, be0=None, resample_every=0):
    '''纯 numpy + Adam 训练 SAE（ReLU+L1 或 TopK）。返回参数 dict（含 act_freq）。'''
    r = np.random.default_rng(seed); n, d = X.shape
    P = dict(We=r.standard_normal((m, d)) * 0.1,
             be=(be0.copy() if be0 is not None else np.zeros(m)),
             bd=X.mean(0).copy())
    Wd = r.standard_normal((d, m)); Wd /= np.linalg.norm(Wd, axis=0, keepdims=True); P['Wd'] = Wd
    mom = {k: np.zeros_like(v) for k, v in P.items()}     # Adam 一阶矩
    vel = {k: np.zeros_like(v) for k, v in P.items()}     # Adam 二阶矩
    freq = np.zeros(m)                                    # 激活频率(EMA)
    for t in range(1, steps + 1):
        idx = r.integers(0, n, size=batch); xb = X[idx]
        # ---- forward ----
        pre = (xb - P['bd']) @ P['We'].T + P['be']
        f = np.maximum(pre, 0.0)
        if activation == 'topk' and K is not None and K < m:
            kth = np.partition(f, -K, axis=1)[:, -K][:, None]
            f = np.where(f >= kth, f, 0.0)
        xhat = f @ P['Wd'].T + P['bd']
        freq = 0.99 * freq + 0.01 * (f > 0).mean(0)
        # ---- backward (手推梯度) ----
        resid = xhat - xb
        g_recon = (2.0 / batch) * resid                   # d MSE / d xhat
        gf = g_recon @ P['Wd']                            # 经解码器回传到 f
        if activation == 'relu':
            gf = gf + (lam / batch) * (f > 0)             # L1 次梯度(只对活的)
        gf = gf * (f > 0)                                 # ReLU/TopK 掩码
        g = dict(We=gf.T @ (xb - P['bd']), Wd=g_recon.T @ f,
                 be=gf.sum(0), bd=g_recon.sum(0))
        # ---- Adam step ----
        for k in P:
            mom[k] = 0.9 * mom[k] + 0.1 * g[k]
            vel[k] = 0.999 * vel[k] + 0.001 * g[k] ** 2
            mhat = mom[k] / (1 - 0.9 ** t); vhat = vel[k] / (1 - 0.999 ** t)
            P[k] -= lr * mhat / (np.sqrt(vhat) + 1e-8)
        P['Wd'] /= (np.linalg.norm(P['Wd'], axis=0, keepdims=True) + 1e-8)  # 解码器列单位化
        # ---- optional resampling（练习3/第5节用） ----
        if resample_every and t % resample_every == 0:
            dead = np.where(freq < 1e-4)[0]
            if len(dead):
                _, xh = sae_forward(X[:2000], P['We'], P['be'], P['Wd'], P['bd'], activation, K)
                err = np.linalg.norm(X[:2000] - xh, axis=1) + 1e-9
                pick = r.choice(2000, size=len(dead), p=err / err.sum())
                v = X[pick] - P['bd']; v /= np.linalg.norm(v, axis=1, keepdims=True) + 1e-8
                P['Wd'][:, dead] = v.T; P['We'][dead] = v; P['be'][dead] = 0.0; freq[dead] = 0.01
                for k in P:                              # 重置该特征的 Adam 状态
                    mom[k] = np.zeros_like(P[k]); vel[k] = np.zeros_like(P[k])
    P['act_freq'] = freq
    return P

sae = train_sae(X, m=80, lam=0.3, steps=4000, activation='relu', seed=1)
f, xhat = sae_forward(X, sae['We'], sae['be'], sae['Wd'], sae['bd'])
mse = np.mean((X - xhat) ** 2); L0 = (f > 0).sum(1).mean()
print(f'L1 SAE 训完: 重建MSE={mse:.4f}  L0={L0:.2f}  活特征比例={(sae["act_freq"]>1e-4).mean():.2f}')
assert mse < 0.3 * np.mean((X - X.mean(0))**2), '应远优于「常数预测」基线'
print('✅ L1 SAE 训练收敛（重建远优于平均基线）')

In [ ]:
def recovery(Wd, F_true, thresh=0.9):
    '''特征恢复：每个真特征找余弦最近的学到字典原子。返回(恢复率, 各真特征最大|cos|, 匹配下标)。'''
    D = Wd / (np.linalg.norm(Wd, axis=0, keepdims=True) + 1e-9)   # 学到的特征方向(列)
    Ft = F_true / (np.linalg.norm(F_true, axis=1, keepdims=True) + 1e-9)
    C = np.abs(Ft @ D)                       # |cos| [m_true, m_learned]
    best = C.max(1); match = C.argmax(1)
    return (best > thresh).mean(), best, match

rate, best, match = recovery(sae['Wd'], F_true, thresh=0.9)
print(f'真特征恢复率(|cos|>0.9) = {rate:.2%}   平均最大余弦 = {best.mean():.3f}')
assert best.mean() > 0.85, 'SAE 应高质量恢复真特征方向'
assert rate > 0.7, '多数真特征应被以高余弦恢复'
# 随机字典基线：与真特征的余弦应低很多
rand_D = rng.standard_normal((20, 80))
_, best_rand, _ = recovery(rand_D, F_true)
print(f'随机字典基线: 平均最大余弦={best_rand.mean():.3f} (应远低于 SAE)')
assert best.mean() > best_rand.mean() + 0.3, 'SAE 应远胜随机字典'
print('✅ 对拍真特征：SAE 学到的字典高质量恢复了植入的真特征，远胜随机')

## 3 · TopK SAE：把 L0 钉死为 K，消除收缩

TopK 编码后只留最大的 `K` 个特征。好处：**L0 恒等于 K**（不用调 λ）、**无 shrinkage**。我们训一个 TopK SAE，验证它的 L0 精确等于 K，且也能恢复真特征。

In [ ]:
K = 4
sae_topk = train_sae(X, m=80, activation='topk', K=K, lr=0.002, steps=4000, seed=2)
f_tk, xh_tk = sae_forward(X, sae_topk['We'], sae_topk['be'], sae_topk['Wd'], sae_topk['bd'],
                          activation='topk', K=K)
L0_tk = (f_tk > 0).sum(1).mean(); mse_tk = np.mean((X - xh_tk)**2)
rate_tk, best_tk, _ = recovery(sae_topk['Wd'], F_true, thresh=0.9)
print(f'TopK(K={K}) SAE: L0={L0_tk:.2f} (应≈{K})  重建MSE={mse_tk:.4f}  平均余弦={best_tk.mean():.3f}')
assert abs(L0_tk - K) < 0.3, 'TopK 的 L0 应精确≈K'
assert best_tk.mean() > 0.75, 'TopK 也应恢复出真特征'
print('✅ TopK 精确控制 L0=K，且恢复真特征 —— 无需调 λ')

## 4 · 重建-稀疏帕累托前沿：怎么比两种 SAE

评 SAE 不能只报一个点。扫一组 λ(L1) 和一组 K(TopK)，各得一条 `(L0, 重建MSE)` 曲线。**更靠左下的前沿更好**（同稀疏度重建更准）。

我们验证两条前沿都**单调**：L0 越大(越不稀疏) 重建越好(MSE 越小)——这就是核心权衡。

In [ ]:
def frontier_L1(lams, m=80, steps=2000):
    pts = []
    for lam in lams:
        s = train_sae(X, m=m, lam=lam, steps=steps, activation='relu', seed=3)
        f, xh = sae_forward(X, s['We'], s['be'], s['Wd'], s['bd'])
        pts.append(((f > 0).sum(1).mean(), np.mean((X - xh)**2)))
    return np.array(pts)

def frontier_topk(Ks, m=80, steps=2000):
    pts = []
    for K in Ks:
        s = train_sae(X, m=m, activation='topk', K=K, steps=steps, seed=3)
        f, xh = sae_forward(X, s['We'], s['be'], s['Wd'], s['bd'], activation='topk', K=K)
        pts.append(((f > 0).sum(1).mean(), np.mean((X - xh)**2)))
    return np.array(pts)

fr_l1 = frontier_L1([0.15, 0.3, 0.6, 1.0])
fr_tk = frontier_topk([2, 3, 5, 8])
print('L1   前沿 (L0, MSE):'); print(np.round(fr_l1, 4))
print('TopK 前沿 (L0, MSE):'); print(np.round(fr_tk, 4))
# 两条前沿都应单调：按 L0 升序排，MSE 应降序
for name, fr in [('L1', fr_l1), ('TopK', fr_tk)]:
    ys = fr[np.argsort(fr[:, 0]), 1]
    assert ys[0] >= ys[-1] - 1e-6, f'{name}: 更稀疏(小L0)应重建更差(大MSE)'
print('✅ 帕累托前沿成立：稀疏度↑↔重建↓。比 SAE = 比谁的前沿更靠左下。')

## 5 · Dead features：诊断与复活

死特征 = 几乎从不激活的字典原子，浪费容量。我们故意用**坏初始化**（编码器偏置很负 → 大量特征 pre<0 → ReLU 恒为 0 → 死）制造死特征，再用 **resampling** 复活，看活特征比例回升。

In [ ]:
m = 80
bad_be = -2.0 * np.ones(m)            # 坏初始化：偏置很负 -> 大量特征死
sae_dead = train_sae(X, m=m, lam=0.3, steps=2000, activation='relu', seed=5,
                     be0=bad_be, resample_every=0)
alive_before = (sae_dead['act_freq'] > 1e-4).mean()
print(f'坏初始化、无 resampling: 活特征比例 = {alive_before:.2%}')

sae_resamp = train_sae(X, m=m, lam=0.3, steps=2000, activation='relu', seed=5,
                       be0=bad_be, resample_every=400)
alive_after = (sae_resamp['act_freq'] > 1e-4).mean()
print(f'坏初始化 + resampling:    活特征比例 = {alive_after:.2%}')
assert alive_after > alive_before + 0.3, 'resampling 应显著提升活特征比例'
print('✅ resampling 把闲置原子重新派去啃最难重建的样本，活特征比例回升')

---
## ✏️ 练习 1：SAE 前向 + 重建质量分解

实现 `sae_eval(x, params, activation, K)`：返回 `(L0, mse, fve)`，其中 `fve`（解释方差比例）`= 1 - ‖x-x̂‖²/‖x-mean(x)‖²`，越接近 1 越好。复用上面的 `sae_forward`。

In [ ]:
def sae_eval(x, params, activation='relu', K=None):
    # TODO: 用 sae_forward 得到 f, xhat；
    #   L0 = 平均每行非零特征数；mse = 平均((x-xhat)**2)；
    #   fvu(未解释方差) = sum((x-xhat)**2)/sum((x-mean)**2)；fve = 1-fvu
    #   返回 (L0, mse, fve)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
L0_, mse_, fve_ = sae_eval(X, sae)        # sae 是第2节训好的 L1 SAE
print(f'L0={L0_:.2f}  mse={mse_:.4f}  解释方差={fve_:.2%}')
assert 0 < L0_ < 80 and mse_ > 0
assert 0.5 < fve_ < 1.0, '训好的 SAE 应解释大部分方差'
_, xh_chk = sae_forward(X, sae['We'], sae['be'], sae['Wd'], sae['bd'])
assert abs(mse_ - np.mean((X-xh_chk)**2)) < 1e-9
print('✅ 练习 1 通过')

## ✏️ 练习 2：L1 vs TopK 的 shrinkage（收缩）

**收缩**：L1 会把*该激活*的特征幅度也压小。实现 `mean_active_magnitude(f)` = 所有非零激活值的平均大小。

比较 L1 与无收缩的 TopK 的平均激活幅度——**L1 应更小（被收缩）**。我们已给好两个训练好的 SAE。

In [ ]:
def mean_active_magnitude(f):
    # TODO: 返回 f 中所有 >0 元素的平均值（衡量激活幅度）；若无非零返回 0.0
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 训一个 L0 与 TopK(K=4) 接近的强 L1 SAE，公平比较激活幅度
sae_l1c = train_sae(X, m=80, lam=0.7, steps=3000, activation='relu', seed=7)
f_l1, _ = sae_forward(X, sae_l1c['We'], sae_l1c['be'], sae_l1c['Wd'], sae_l1c['bd'])
f_tkc, _ = sae_forward(X, sae_topk['We'], sae_topk['be'], sae_topk['Wd'], sae_topk['bd'],
                       activation='topk', K=4)
mag_l1 = mean_active_magnitude(f_l1); mag_tk = mean_active_magnitude(f_tkc)
print(f'L1(lam=0.7) L0={ (f_l1>0).sum(1).mean():.2f} 平均激活幅度={mag_l1:.3f}')
print(f'TopK(K=4)   L0={ (f_tkc>0).sum(1).mean():.2f} 平均激活幅度={mag_tk:.3f}')
assert mag_l1 > 0 and mag_tk > 0
assert mag_l1 < mag_tk, 'L1 的激活被收缩 -> 平均幅度应小于无收缩的 TopK'
print('✅ 练习 2 通过：亲眼看到 L1 的 shrinkage（幅度被压小）')

## ✏️ 练习 3：手写 dead-feature resampling

实现 `resample_dead(X, params, freq, eps=1e-4, seed=0)`：找出 `freq<eps` 的死特征，按**重建误差**加权采样同样多的样本，把这些样本（去 bd、行单位化）写进死特征的解码器列 `Wd[:,dead]` 与编码器行 `We[dead]`，并把 `be[dead]=0`。返回 `(新params, 复活特征数)`。

In [ ]:
def resample_dead(X, params, freq, eps=1e-4, seed=0):
    r = np.random.default_rng(seed)
    We, be, Wd, bd = params['We'].copy(), params['be'].copy(), params['Wd'].copy(), params['bd'].copy()
    dead = np.where(freq < eps)[0]
    # TODO: 若有死特征：
    #   1) 取前2000样本算重建误差 err=‖x-xhat‖ (用 sae_forward)
    #   2) p=err/err.sum()，按 p 采样 len(dead) 个样本下标 pick
    #   3) v=X[pick]-bd; 行单位化; Wd[:,dead]=v.T; We[dead]=v; be[dead]=0
    raise NotImplementedError
    return dict(We=We, be=be, Wd=Wd, bd=bd), len(dead)

In [ ]:
# —— 练习 3 自测 ——
# 制造死特征：把第2节SAE的前40个特征手动'杀死'(置零解码列并标记freq=0)
params0 = {k: sae[k].copy() for k in ('We','be','Wd','bd')}
freq0 = sae['act_freq'].copy()
kill = np.arange(40)
freq0[kill] = 0.0
params0['Wd'][:, kill] = 0.0
new_params, n_revived = resample_dead(X, params0, freq0)
print(f'复活特征数 = {n_revived}')
assert n_revived == 40
norms = np.linalg.norm(new_params['Wd'][:, kill], axis=0)
assert np.allclose(norms, 1.0, atol=1e-6), '复活的解码器列应单位范数'
print('✅ 练习 3 通过：死特征被重置到高误差样本方向并单位化')

## ✏️ 练习 4：重建质量随过完备度变化

更宽的字典（更大 `m`）能容纳更多特征 → 重建更好。实现 `mse_vs_width(widths, steps=2000)`：对每个宽度训一个 L1 SAE，返回 `[(m, mse), ...]`。验证：宽度越大，重建 MSE 越小（单调不增）。

In [ ]:
def mse_vs_width(widths, steps=2000, lam=0.3):
    # TODO: 对每个 m in widths：train_sae(X, m=m, lam=lam, steps=steps, seed=11)
    #       前向算 mse，收集 (m, mse)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
res = mse_vs_width([20, 40, 80])
print('(width, mse):', [(m, round(e,4)) for m,e in res])
mses = [e for _, e in res]
assert mses[0] >= mses[-1] - 1e-3, '更宽的字典重建应不更差'
print('✅ 练习 4 通过：过完备度越高，重建越好（容量换保真）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def sae_eval(x, params, activation='relu', K=None):
    f, xhat = sae_forward(x, params['We'], params['be'], params['Wd'], params['bd'], activation, K)
    L0 = (f > 0).sum(1).mean()
    mse = np.mean((x - xhat) ** 2)
    fvu = np.sum((x - xhat) ** 2) / np.sum((x - x.mean(0)) ** 2)
    return L0, mse, 1.0 - fvu

In [ ]:
# 练习 2 参考答案
def mean_active_magnitude(f):
    active = f[f > 0]
    return float(active.mean()) if active.size else 0.0

In [ ]:
# 练习 3 参考答案
def resample_dead(X, params, freq, eps=1e-4, seed=0):
    r = np.random.default_rng(seed)
    We, be, Wd, bd = params['We'].copy(), params['be'].copy(), params['Wd'].copy(), params['bd'].copy()
    dead = np.where(freq < eps)[0]
    if len(dead):
        _, xh = sae_forward(X[:2000], We, be, Wd, bd)
        err = np.linalg.norm(X[:2000] - xh, axis=1) + 1e-9
        pick = r.choice(2000, size=len(dead), p=err / err.sum())
        v = X[pick] - bd
        v /= (np.linalg.norm(v, axis=1, keepdims=True) + 1e-8)
        Wd[:, dead] = v.T; We[dead] = v; be[dead] = 0.0
    return dict(We=We, be=be, Wd=Wd, bd=bd), len(dead)

In [ ]:
# 练习 4 参考答案
def mse_vs_width(widths, steps=2000, lam=0.3):
    out = []
    for m in widths:
        s = train_sae(X, m=m, lam=lam, steps=steps, seed=11)
        _, xh = sae_forward(X, s['We'], s['be'], s['Wd'], s['bd'])
        out.append((m, float(np.mean((X - xh) ** 2))))
    return out

---
## 🧪 真实数据胶囊：真实 SAE 的配置与规模

下面是几个**真实公开 SAE** 的配置（来自论文 / Neuronpedia 的公开数字，约数）。用它们算**过完备倍数**与字典规模，体会真实 SAE 的体量——你刚从零训的玩具，和它们是同一套机器，只是大了几个数量级。

In [ ]:
# 真实 SAE 公开配置（约数；d_model=隐藏维, n_features=SAE宽度, L0=平均激活特征数）
REAL_SAES = {
    'GPT-2 small (J.Bloom, res)':       dict(d_model=768,   n_features=24576,      L0=40,  kind='L1'),
    'Gemma-2-2B (Gemma Scope, res)':    dict(d_model=2304,  n_features=16384,      L0=70,  kind='JumpReLU'),
    'Claude 3 Sonnet (Anthropic, 34M)': dict(d_model=4096,  n_features=34_000_000, L0=300, kind='L1'),
}
print(f"{'SAE':34s}{'过完备×':>9s}{'特征数':>13s}{'L0':>6s}{'类型':>11s}")
for name, s in REAL_SAES.items():
    ratio = s['n_features'] / s['d_model']
    print(f"{name:34s}{ratio:>8.1f}x{s['n_features']:>13,d}{s['L0']:>6d}{s['kind']:>11s}")
print('\n观察：过完备倍数从~8x(小模型)到数千x(Sonnet 34M/4096≈8300x)；L0 都远小于特征数(高度稀疏)。')
print('你的玩具 SAE(80/20=4x, L0≈几) 与它们结构相同，只差规模。')

**🧪 胶囊练习**：实现 `bytes_for_sae(d_model, n_features, dtype_bytes=2)`：估算一个 SAE 编码器+解码器权重的存储字节数（两个 `d_model × n_features` 矩阵），返回字节数。用它算 Claude 3 Sonnet 34M 特征 SAE 的权重有多大（GB）。

In [ ]:
def bytes_for_sae(d_model, n_features, dtype_bytes=2):
    # TODO: 编码器(n_features×d_model)+解码器(d_model×n_features)=2*d_model*n_features 个参数
    #       返回 总字节数 = 参数数 * dtype_bytes
    raise NotImplementedError

In [ ]:
# 自测
b = bytes_for_sae(4096, 34_000_000, dtype_bytes=2)
gb = b / 1e9
assert b == 2 * 4096 * 34_000_000 * 2
print(f'Claude 3 Sonnet 34M 特征 SAE 权重 ≈ {gb:.0f} GB (fp16)')
assert gb > 100, '应是数百 GB 量级 —— 前沿 SAE 的真实成本'
print('✅ 胶囊练习通过：前沿 SAE 的字典本身就是几百 GB 的大模型')

In [ ]:
# 📖 胶囊参考答案
def bytes_for_sae(d_model, n_features, dtype_bytes=2):
    n_params = 2 * d_model * n_features
    return n_params * dtype_bytes

### 小结
- **SAE = 特征解压器**：编码到过完备稀疏空间、解码回激活；解码器列 = 字典原子(特征方向, 单位范数)。
- **核心张力**：重建 vs 稀疏(L0)，比 SAE = 比谁的**帕累托前沿**更靠左下。
- **三代**：L1(有 shrinkage) → TopK(L0=K, 无收缩, 易死, 用 auxk) → JumpReLU(可学阈值+罚 L0, 常最优)。
- **训练实操**：解码器列每步单位化、用 **Adam**(SGD 收敛差)、监控并 resample **dead features**。
- **合成数据独有的检验**：用真特征 F 对拍**恢复率**(本课 L1 恢复 >90%)，校准好再上真模型。

下一站：**模块 02 · 单义特征与叠加** —— 拆开之后，单个特征到底长什么样、怎么知道拆干净了。